# PyLance Variation Dataset Builder

PyLance sandbox for building a Lance variation dataset from the merged VEPyr Parquet cache. It applies Arrow field metadata such as `{"lance-encoding:compression": "fsst"}`, derives chr-local `position UInt32` from `start`, adds a warm/cold `tier`, creates scalar indexes, and invokes the Rust sandbox CLI to render real Lance metadata and observed encoding tables with `rich`.

Environment setup from the repo root:

```bash
uv venv research/lance_encoding_sandbox/.venv-pylance
uv pip install --python research/lance_encoding_sandbox/.venv-pylance/bin/python \
  -r research/lance_encoding_sandbox/requirements-pylance.txt
research/lance_encoding_sandbox/.venv-pylance/bin/python -m ipykernel install \
  --user --name lance-pylance-sandbox --display-name "Lance PyLance Sandbox"
```

Select the `Lance PyLance Sandbox` kernel before running the notebook.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shutil
import subprocess
import time
import tomllib
from typing import Any, Iterable, Iterator, Mapping

import lance
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import Markdown, display
from rich.console import Console
from rich.table import Table

console = Console(width=180)
print('pylance/lance', getattr(lance, '__version__', '<unknown>'))
print('pyarrow', pa.__version__)


pylance/lance 7.0.0
pyarrow 24.0.0


In [2]:
# ----- Build configuration -----
REPO = Path('/Users/mwiewior/research/git/datafusion-bio-functions')
CACHE_ROOT = Path('/Users/mwiewior/workspace/data_vepyr/115_GRCh38_merged')
VARIATION_DIR = CACHE_ROOT / 'variation'
OUTPUT_ROOT = Path('/Users/mwiewior/workspace/data_vepyr/lance_encoding_sandbox')

CHROM = 'chr1'
WARM_PARQUET = VARIATION_DIR / f'{CHROM}_warm.parquet'
COLD_PARQUET = VARIATION_DIR / f'{CHROM}_cold.parquet'
STRUCT_PACKING_ENABLED = False
LANCE_VERSION = '2.2' if STRUCT_PACKING_ENABLED else '2.1'
DATASET_PROFILE = 'v22_packed_targeted' if STRUCT_PACKING_ENABLED else 'v21'
DATASET_NAME = f'pylance_variation_{CHROM}_{DATASET_PROFILE}_position_u32'
OUTPUT_DATASET = OUTPUT_ROOT / DATASET_NAME / f'{CHROM}.lance'
REPORTS_DIR = OUTPUT_ROOT / DATASET_NAME / 'reports'
RUST_SANDBOX_MANIFEST = REPO / 'research/lance_encoding_sandbox/crates/lance_sandbox/Cargo.toml'
# Global Lance primitive miniblock value cap used by the in-process PyLance writer.
# Set to None to remove the env var and use Lance's built-in default.
LANCE_MINIBLOCK_MAX_VALUES: str | None = '4096'
# PyLance hands input batches to the Lance writer. Keep this at a modest
# value for nested/list fields while still avoiding tiny Python batches.
PARQUET_BATCH_SIZE = 16_384
OVERWRITE = True
# Write warm and cold tiers in separate Lance transactions. This keeps tier
# fragments physically separated and avoids a Lance 2.1 boundary failure when
# variant_keys switches from real warm lists to all-null cold lists.
WRITE_TIERS_SEPARATELY = True

# Safety switches. Set these to True when you want to build/index the full dataset.
RUN_BUILD = True
RUN_INDEXES = True

# Smoke-test control. Use an integer such as 10_000 before running a full chr1 build.
ROW_LIMIT_PER_TIER: int | None = None

# The chr-local Lance layout uses position UInt32 from start; position_key is not needed.
DROP_COLUMNS = {'position_key'}

# Defaults are applied to every field unless FIELD_METADATA overrides a key.
DEFAULT_FIELD_METADATA: dict[str, str] = {
    'lance-encoding:structural-encoding': 'miniblock',
    'lance-encoding:compression': 'zstd',
    'lance-encoding:compression-level': '3',
    'lance-encoding:dict-values-compression': 'zstd',
    'lance-encoding:dict-values-compression-level': '3',
    'lance-encoding:rle-threshold': '0.95',
    'lance-encoding:dict-size-ratio': '0.99',
    'lance-encoding:dict-divisor': '1',
    'lance-encoding:minichunk-size': f'{str(4096)}',
}

# Per-field overrides. Uncomment or add fields for experiments.
FIELD_METADATA: dict[str, dict[str, str | None]] = {
    # 'variation_name': {'lance-encoding:compression': 'fsst'},
    # 'clin_sig': {'lance-encoding:compression': 'fsst'},
    # 'start': {'lance-encoding:bss': 'on'},
    # 'end': {'lance-encoding:bss': 'on'},
}

STRUCT_PACKING_GROUPS: dict[str, list[str]] = {
    'match_payload': ['allele_string', 'end', 'failed'],
    'identity_text': ['variation_name', 'dbsnp_ids'],
    'clinical_payload': ['clin_sig', 'clin_sig_allele', 'clinical_impact', 'pubmed', 'clinvar_ids', 'cosmic_ids'],
    'variant_flags': ['somatic', 'phenotype_or_disease', 'strand'],
    'freq_1kg': ['AF', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS'],
    'freq_gnomade': ['gnomADe', 'gnomADe_AFR', 'gnomADe_AMR', 'gnomADe_ASJ', 'gnomADe_EAS', 'gnomADe_FIN', 'gnomADe_NFE', 'gnomADe_REMAINING', 'gnomADe_SAS', 'gnomADe_MID'],
    'freq_gnomadg_core': ['gnomADg', 'gnomADg_AFR', 'gnomADg_NFE', 'gnomADg_AMR'],
    'freq_gnomadg_tail': ['gnomADg_AMI', 'gnomADg_ASJ', 'gnomADg_EAS', 'gnomADg_FIN', 'gnomADg_MID', 'gnomADg_SAS', 'gnomADg_REMAINING'],
}

INDEX_DEFINITIONS = [
    {'column': 'position', 'index_type': 'BTREE', 'name': 'position_btree_idx'},
    {'column': 'tier', 'index_type': 'BITMAP', 'name': 'tier_bitmap_idx'},
]


In [3]:
# ----- Schema and streaming helpers -----
def _metadata_to_bytes(metadata: Mapping[str, str | None] | None) -> dict[bytes, bytes] | None:
    if not metadata:
        return None
    return {str(k).encode(): str(v).encode() for k, v in metadata.items() if v is not None}


def _metadata_to_str(metadata: Mapping[bytes | str, bytes | str] | None) -> dict[str, str]:
    out: dict[str, str] = {}
    for k, v in (metadata or {}).items():
        key = k.decode() if isinstance(k, bytes) else str(k)
        value = v.decode() if isinstance(v, bytes) else str(v)
        out[key] = value
    return out


def metadata_for_field(name: str) -> dict[str, str]:
    merged: dict[str, str | None] = dict(DEFAULT_FIELD_METADATA)
    merged.update(FIELD_METADATA.get(name, {}))
    return {str(k): str(v) for k, v in merged.items() if v is not None}


def metadata_for_struct(name: str) -> dict[str, str]:
    return {
        'lance-encoding:structural-encoding': metadata_for_field(name).get('lance-encoding:structural-encoding', 'miniblock'),
        'lance-encoding:packed': 'true',
    }


def apply_metadata(field: pa.Field, *, nullable: bool | None = None) -> pa.Field:
    return pa.field(
        field.name,
        field.type,
        nullable=field.nullable if nullable is None else nullable,
        metadata=_metadata_to_bytes(metadata_for_field(field.name)),
    )


def apply_struct_metadata(field: pa.Field) -> pa.Field:
    return pa.field(field.name, field.type, nullable=field.nullable, metadata=_metadata_to_bytes(metadata_for_struct(field.name)))


def enabled_struct_groups() -> dict[str, list[str]]:
    if not STRUCT_PACKING_ENABLED:
        return {}
    return {
        parent: [child for child in children if child not in DROP_COLUMNS]
        for parent, children in STRUCT_PACKING_GROUPS.items()
    }


def child_to_struct() -> dict[str, str]:
    mapping: dict[str, str] = {}
    for parent, children in enabled_struct_groups().items():
        for child in children:
            if child in mapping:
                raise ValueError(f'field {child!r} appears in both {mapping[child]!r} and {parent!r}')
            mapping[child] = parent
    return mapping


def build_target_schema(warm_path: Path, cold_path: Path) -> pa.Schema:
    warm_schema = pq.read_schema(warm_path)
    cold_schema = pq.read_schema(cold_path)
    cold_names = set(cold_schema.names)
    warm_fields = {field.name: field for field in warm_schema}
    groups = enabled_struct_groups()
    child_parent = child_to_struct()
    emitted_structs: set[str] = set()
    fields: list[pa.Field] = []
    for field in warm_schema:
        if field.name in DROP_COLUMNS:
            continue
        parent = child_parent.get(field.name)
        if parent is not None:
            if parent in emitted_structs:
                continue
            child_fields: list[pa.Field] = []
            for child in groups[parent]:
                child_field = warm_fields.get(child)
                if child_field is None:
                    continue
                nullable = child_field.nullable or child not in cold_names
                child_fields.append(apply_metadata(child_field, nullable=nullable))
            if child_fields:
                fields.append(apply_struct_metadata(pa.field(parent, pa.struct(child_fields), nullable=True)))
                emitted_structs.add(parent)
            continue
        # Cold chr parquet does not contain variant_keys; make any missing warm field nullable.
        nullable = field.nullable or field.name not in cold_names
        fields.append(apply_metadata(field, nullable=nullable))
    fields.append(apply_metadata(pa.field('tier', pa.int8(), nullable=False)))
    fields.append(apply_metadata(pa.field('position', pa.uint32(), nullable=False)))
    return pa.schema(fields)


TARGET_SCHEMA = build_target_schema(WARM_PARQUET, COLD_PARQUET)
print('fields', len(TARGET_SCHEMA))
print('struct packing', STRUCT_PACKING_ENABLED, 'lance_version', LANCE_VERSION)
print('first fields', TARGET_SCHEMA.names[:8])
print('derived fields present', {'tier': TARGET_SCHEMA.get_field_index('tier'), 'position': TARGET_SCHEMA.get_field_index('position')})


fields 80
first fields ['chrom', 'start', 'end', 'variation_name', 'allele_string', 'failed', 'somatic', 'strand']
derived fields present {'tier': 78, 'position': 79}


In [4]:
def array_for_schema_field(source: Mapping[str, pa.Array], field: pa.Field, row_count: int) -> pa.Array:
    if field.name == 'tier':
        return pa.array([0] * row_count, type=field.type)
    if field.name == 'position':
        if 'start' not in source:
            raise ValueError('source batch does not contain start; cannot derive position')
        return pc.cast(source['start'], field.type, safe=True)
    if field.name in source:
        arr = source[field.name]
        return arr if arr.type == field.type else pc.cast(arr, field.type, safe=True)
    return pa.nulls(row_count, type=field.type)


def struct_array_for_field(source: Mapping[str, pa.Array], field: pa.Field, row_count: int) -> pa.StructArray:
    child_fields = list(field.type)
    child_arrays = [array_for_schema_field(source, child, row_count) for child in child_fields]
    return pa.StructArray.from_arrays(child_arrays, fields=child_fields)


def transform_batch(batch: pa.RecordBatch, *, tier: int, schema: pa.Schema) -> pa.RecordBatch:
    source = {name: batch.column(i) for i, name in enumerate(batch.schema.names)}
    arrays: list[pa.Array] = []
    for field in schema:
        if field.name == 'tier':
            arrays.append(pa.array([tier] * batch.num_rows, type=field.type))
        elif pa.types.is_struct(field.type):
            arrays.append(struct_array_for_field(source, field, batch.num_rows))
        else:
            arrays.append(array_for_schema_field(source, field, batch.num_rows))
    return pa.RecordBatch.from_arrays(arrays, schema=schema)


def iter_transformed_batches_for_tier(path: Path, *, tier: int, schema: pa.Schema) -> Iterator[pa.RecordBatch]:
    remaining = ROW_LIMIT_PER_TIER
    reader = pq.ParquetFile(path)
    for batch in reader.iter_batches(batch_size=PARQUET_BATCH_SIZE, use_threads=True):
        if remaining is not None:
            if remaining <= 0:
                break
            batch = batch.slice(0, min(batch.num_rows, remaining))
            remaining -= batch.num_rows
        if batch.num_rows:
            yield transform_batch(batch, tier=tier, schema=schema)


def iter_transformed_batches(schema: pa.Schema) -> Iterator[pa.RecordBatch]:
    for path, tier in [(WARM_PARQUET, 0), (COLD_PARQUET, 1)]:
        yield from iter_transformed_batches_for_tier(path, tier=tier, schema=schema)


def configure_lance_writer_environment() -> None:
    if LANCE_MINIBLOCK_MAX_VALUES is None:
        os.environ.pop('LANCE_MINIBLOCK_MAX_VALUES', None)
        print('LANCE_MINIBLOCK_MAX_VALUES unset; Lance default applies')
    else:
        os.environ['LANCE_MINIBLOCK_MAX_VALUES'] = str(LANCE_MINIBLOCK_MAX_VALUES)
        print(f'LANCE_MINIBLOCK_MAX_VALUES={os.environ["LANCE_MINIBLOCK_MAX_VALUES"]}')


def build_lance_dataset() -> lance.LanceDataset:
    configure_lance_writer_environment()
    if OUTPUT_DATASET.exists() and OVERWRITE:
        shutil.rmtree(OUTPUT_DATASET)
    OUTPUT_DATASET.parent.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    mode = 'overwrite' if OVERWRITE else 'create'
    if WRITE_TIERS_SEPARATELY:
        ds: lance.LanceDataset | None = None
        for label, path, tier in [('warm', WARM_PARQUET, 0), ('cold', COLD_PARQUET, 1)]:
            phase_start = time.perf_counter()
            reader = pa.RecordBatchReader.from_batches(
                TARGET_SCHEMA,
                iter_transformed_batches_for_tier(path, tier=tier, schema=TARGET_SCHEMA),
            )
            ds = lance.write_dataset(
                reader,
                OUTPUT_DATASET,
                schema=TARGET_SCHEMA,
                mode=mode,
                data_storage_version=LANCE_VERSION,
            )
            print(f'wrote {label} tier in {time.perf_counter() - phase_start:.3f}s')
            mode = 'append'
        if ds is None:
            raise RuntimeError('no tiers were written')
        print(f'wrote {OUTPUT_DATASET} in {time.perf_counter() - start:.3f}s')
        return lance.dataset(OUTPUT_DATASET)

    reader = pa.RecordBatchReader.from_batches(TARGET_SCHEMA, iter_transformed_batches(TARGET_SCHEMA))
    ds = lance.write_dataset(
        reader,
        OUTPUT_DATASET,
        schema=TARGET_SCHEMA,
        mode=mode,
        data_storage_version=LANCE_VERSION,
    )
    print(f'wrote {OUTPUT_DATASET} in {time.perf_counter() - start:.3f}s')
    return ds


if RUN_BUILD:
    dataset = build_lance_dataset()
else:
    print('RUN_BUILD is False; set it to True to create the Lance dataset.')


LANCE_MINIBLOCK_MAX_VALUES=4096


[2026-06-13T13:39:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/mwiewior/workspace/data_vepyr/lance_encoding_sandbox/pylance_variation_chr1_v21_position_u32/chr1.lance, it will be created


wrote warm tier in 4.941s
wrote cold tier in 58.740s
wrote /Users/mwiewior/workspace/data_vepyr/lance_encoding_sandbox/pylance_variation_chr1_v21_position_u32/chr1.lance in 63.681s


In [5]:
# ----- Scalar indexes -----
def is_lance_dataset(path: Path) -> bool:
    return path.exists() and (path / '_versions').exists()


def open_dataset() -> lance.LanceDataset:
    if not is_lance_dataset(OUTPUT_DATASET):
        raise FileNotFoundError(
            f'{OUTPUT_DATASET} is not a valid Lance dataset. '
            'Set RUN_BUILD=True with OVERWRITE=True to replace a partial build.'
        )
    return lance.dataset(OUTPUT_DATASET)


def create_scalar_indexes() -> lance.LanceDataset:
    ds = open_dataset()
    for spec in INDEX_DEFINITIONS:
        print(f"creating {spec['index_type']} index {spec['name']} on {spec['column']}")
        ds.create_scalar_index(
            spec['column'],
            spec['index_type'],
            name=spec['name'],
            replace=True,
        )
        ds = lance.dataset(OUTPUT_DATASET)
    return ds


if RUN_INDEXES:
    dataset = create_scalar_indexes()
else:
    print('RUN_INDEXES is False; set it to True after building to create BTREE(position) and BITMAP(tier).')


creating BTREE index position_btree_idx on position
creating BITMAP index tier_bitmap_idx on tier


In [ ]:
# ----- Python/Rich inspection -----
def human_bytes(n: int | None) -> str:
    if n is None:
        return 'n/a'
    value = float(n)
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if abs(value) < 1024 or unit == 'TiB':
            return f'{value:.2f} {unit}' if unit != 'B' else f'{int(value)} B'
        value /= 1024
    return f'{n} B'


def rich_table(title: str, rows: list[dict[str, Any]], columns: list[str], show_lines: bool = False) -> None:
    table = Table(title=title, show_lines=show_lines)
    for column in columns:
        justify = 'right' if column.endswith('_bytes') or column in {'rows', 'fragments', 'files'} else 'left'
        table.add_column(column, justify=justify, overflow='fold')
    for row in rows:
        table.add_row(*[str(row.get(column, '')) for column in columns])
    console.print(table)


ENCODING_METADATA_KEYS = [
    'lance-encoding:structural-encoding',
    'lance-encoding:compression',
    'lance-encoding:compression-level',
    'lance-encoding:dict-values-compression',
    'lance-encoding:dict-values-compression-level',
    'lance-encoding:rle-threshold',
    'lance-encoding:dict-size-ratio',
    'lance-encoding:dict-divisor',
    'lance-encoding:minichunk-size',
    'lance-encoding:packed',
]


def run_rust_inspect() -> dict[str, Any]:
    command = [
        'cargo', 'run', '--release',
        '--manifest-path', str(RUST_SANDBOX_MANIFEST),
        '--', 'inspect-path',
        '--dataset-path', str(OUTPUT_DATASET),
        '--lance-version', LANCE_VERSION,
        '--input-parquet', str(WARM_PARQUET),
        '--input-parquet', str(COLD_PARQUET),
        '--position-field', 'position',
        '--position-source-column', 'start',
    ]
    command_text = ' '.join(command)
    console.print(f'Running Rust inspect: {command_text}')
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            'Rust Lance inspect failed.\n'
            f'Command: {command_text}\n'
            f'stdout:\n{result.stdout}\n'
            f'stderr:\n{result.stderr}'
        )
    return json.loads(result.stdout)


def metadata_summary(metadata: Mapping[str, Any]) -> str:
    parts = [f'{key}={metadata[key]}' for key in ENCODING_METADATA_KEYS if key in metadata]
    return '; '.join(parts)


def rust_schema_rows(report: Mapping[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for field in report.get('logical_fields', []):
        metadata = field.get('metadata') or {}
        input_parquet = field.get('input_parquet') or {}
        rows.append({
            'field': field.get('path', ''),
            'id': field.get('id', ''),
            'data_type': field.get('data_type', ''),
            'nullable': field.get('nullable', ''),
            'lance_compressed': human_bytes(field.get('compressed_bytes')),
            'input_parquet': human_bytes(input_parquet.get('compressed_bytes')),
            'input_parquet_column_chunks': input_parquet.get('column_chunks', 0),
            'input_parquet_encodings': ','.join(input_parquet.get('encodings') or []),
            'input_parquet_compression': ','.join(input_parquet.get('compression') or []),
            'pages': field.get('pages', 0),
            'metadata': metadata_summary(metadata),
            'observed_encodings': '\n'.join(observed_encoding_label(value) for value in (field.get('encodings_observed') or [])),
        })
    return rows


def observed_encoding_label(value: Any) -> str:
    if isinstance(value, Mapping):
        return 'layout={}; encoding={}; compression={}'.format(
            value.get('layout', ''),
            value.get('encoding', ''),
            value.get('compression', ''),
        )
    return str(value)


def rust_index_rows(report: Mapping[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for index in report.get('indexes', []):
        files = index.get('files') or {}
        size_bytes = index.get('size_bytes')
        rows.append({
            'name': index.get('name', ''),
            'fields': ','.join(index.get('fields') or []),
            'dataset_version': index.get('dataset_version', ''),
            'index_version': index.get('index_version', ''),
            'size': human_bytes(size_bytes),
            'size_bytes': size_bytes,
            'files': len(files),
            'stats': index.get('statistics_json') or '',
        })
    return rows


def rust_size_rows(report: Mapping[str, Any]) -> list[dict[str, Any]]:
    rows = [
        ('data', report.get('data_size_bytes', 0), ''),
        ('indices', report.get('index_size_bytes', 0), ''),
        ('metadata', report.get('metadata_size_bytes', 0), ''),
        ('other', report.get('other_size_bytes', 0), ''),
        ('total', report.get('total_size_bytes', 0), report.get('file_count', '')),
    ]
    return [{'class': name, 'size': human_bytes(size), 'size_bytes': size, 'files': files} for name, size, files in rows]


def inspect_fragment_rows(ds: lance.LanceDataset) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for fragment in ds.get_fragments():
        for data_file in fragment.data_files():
            rows.append({
                'fragment': fragment.fragment_id,
                'rows': fragment.physical_rows,
                'file': data_file.path,
                'version': f'{data_file.file_major_version}.{data_file.file_minor_version}',
                'fields': len(data_file.field_ids()),
                'size': human_bytes(data_file.file_size_bytes),
                'size_bytes': data_file.file_size_bytes,
            })
    return rows


def inspect_size_rows(dataset_path: Path) -> list[dict[str, Any]]:
    classes = {'data': 0, 'indices': 0, 'metadata': 0, 'other': 0}
    file_count = 0
    for path in dataset_path.rglob('*'):
        if not path.is_file():
            continue
        file_count += 1
        size = path.stat().st_size
        rel = path.relative_to(dataset_path).as_posix()
        if '_indices' in rel or 'indices' in rel:
            classes['indices'] += size
        elif rel.endswith('.lance'):
            classes['data'] += size
        elif rel.startswith('_versions') or rel.startswith('_transactions') or rel.endswith('.manifest'):
            classes['metadata'] += size
        else:
            classes['other'] += size
    rows = [{'class': key, 'size': human_bytes(value), 'size_bytes': value, 'files': ''} for key, value in classes.items()]
    rows.append({'class': 'total', 'size': human_bytes(sum(classes.values())), 'size_bytes': sum(classes.values()), 'files': file_count})
    return rows


In [ ]:
def inspect_dataset_with_rust_cli() -> dict[str, Any]:
    ds = open_dataset()
    rust_report = run_rust_inspect()
    schema_rows = rust_schema_rows(rust_report)
    index_rows = rust_index_rows(rust_report)
    fragment_rows = inspect_fragment_rows(ds)
    size_rows = rust_size_rows(rust_report)

    rich_table('Dataset Sizes', size_rows, ['class', 'size', 'size_bytes', 'files'])
    rich_table('Scalar Indexes', index_rows, ['name', 'fields', 'dataset_version', 'index_version', 'size', 'size_bytes', 'files'])
    rich_table('Fragments / Data Files', fragment_rows[:40], ['fragment', 'rows', 'file', 'version', 'fields', 'size', 'size_bytes'])
    rich_table('Field Metadata / Observed Encodings', schema_rows, ['field', 'id', 'data_type', 'nullable', 'lance_compressed', 'input_parquet', 'input_parquet_encodings', 'input_parquet_compression', 'pages', 'metadata', 'observed_encodings'], show_lines=True)

    display(pd.DataFrame(schema_rows))
    display(pd.DataFrame(index_rows))
    display(pd.DataFrame(fragment_rows))

    report = {
        'dataset_path': str(OUTPUT_DATASET),
        'lance_version': LANCE_VERSION,
        'rows': ds.count_rows(),
        'schema': schema_rows,
        'indexes': index_rows,
        'fragments': fragment_rows,
        'sizes': size_rows,
        'rust_inspect': rust_report,
        'note': 'Field metadata, compressed bytes, pages, and observed encodings are produced by the Rust lance_sandbox inspect-path command.',
    }
    return report


if is_lance_dataset(OUTPUT_DATASET):
    inspect_report = inspect_dataset_with_rust_cli()
else:
    inspect_report = None
    print(f'{OUTPUT_DATASET} is not a valid Lance dataset yet. Set RUN_BUILD=True to build it.')


In [ ]:
# ----- 10k lookup benchmark via Rust CLI -----
BENCHMARK_CONFIG = REPO / ('research/lance_encoding_sandbox/configs/packed_targeted_v22.toml' if STRUCT_PACKING_ENABLED else 'research/lance_encoding_sandbox/configs/current_v21_zstd3.toml')
BENCHMARK_10K_POSITIONS = REPO / 'research/lance_encoding_sandbox/inputs/chr1_cold_sample_10k_positions_u32.txt'
BENCHMARK_RUSTFLAGS = '-C target-cpu=native'
BENCHMARK_PLAN_MAX_LINE_CHARS = 240
BENCHMARK_FILTER_MAX_CHARS = 220


def benchmark_human_bytes(n: int | None) -> str:
    if n is None:
        return 'n/a'
    value = float(n)
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if abs(value) < 1024 or unit == 'TiB':
            return f'{int(value)} B' if unit == 'B' else f'{value:.2f} {unit}'
        value /= 1024
    return f'{n} B'


def benchmark_truncate_line(line: str, max_chars: int) -> str:
    if len(line) <= max_chars:
        return line
    keep = max(16, max_chars - 40)
    omitted = len(line) - keep
    marker = f' ... <truncated {omitted} chars>'
    keep = max(16, max_chars - len(marker))
    omitted = len(line) - keep
    return line[:keep] + f' ... <truncated {omitted} chars>'


def benchmark_plan_text_for_display(plan_text: str) -> str:
    return '\n'.join(benchmark_truncate_line(line, BENCHMARK_PLAN_MAX_LINE_CHARS) for line in plan_text.splitlines())


def benchmark_report_path(config_path: Path, dataset_path: Path | None = None) -> Path:
    if dataset_path is not None:
        return dataset_path.parent / 'reports' / 'benchmark.json'
    config = tomllib.loads(config_path.read_text())
    dataset = config['dataset']
    return Path(dataset['output_root']) / dataset['name'] / 'reports' / 'benchmark.json'


def scan_result_markdown_row(result: Mapping[str, Any]) -> str:
    seconds = float(result.get('seconds') or 0.0)
    rows = int(result.get('rows') or 0)
    rows_per_second = int(rows / seconds) if seconds else 0
    io = result.get('io') or {}
    values = [
        result.get('name', ''),
        result.get('lookup_batch_size') or 0,
        result.get('scans', 0),
        rows,
        result.get('selected_positions', 0),
        f'{seconds:.6f}',
        rows_per_second,
        benchmark_human_bytes(io.get('bytes_read', 0)),
        io.get('iops', 0),
        io.get('requests', 0),
        io.get('ranges_scanned', 0),
        io.get('fragments_scanned', 0),
    ]
    return '| ' + ' | '.join(str(value) for value in values) + ' |'


def benchmark_physical_plans_markdown(report: Mapping[str, Any]) -> str:
    plans = report.get('physical_plans') or []
    if not plans:
        return ''
    sections = ['## Physical Plans']
    for plan in plans:
        plan_text = benchmark_plan_text_for_display(str(plan.get('plan') or '').rstrip())
        filter_text = benchmark_truncate_line(str(plan.get('filter', '')).replace('`', '\\`'), BENCHMARK_FILTER_MAX_CHARS)
        sections.append('\n'.join([
            f'### {plan.get("name", "")}',
            '',
            f'- Batch keys: `{plan.get("lookup_batch_size") or 0}`',
            f'- Projected fields: `{plan.get("projected_column_count", "")}`',
            f'- Filter: `{filter_text}`',
            '',
            '```text',
            plan_text,
            '```',
        ]))
    return '\n\n'.join(sections)


def benchmark_markdown(report: Mapping[str, Any], *, config_path: Path, positions_file: Path, command: list[str]) -> str:
    headers = [
        'Path', 'Batch keys', 'Scans', 'Rows', 'Selected positions', 'Seconds',
        'Rows/s', 'Bytes read', 'IOPs', 'Requests', 'Ranges scanned', 'Fragments scanned',
    ]
    rows = [scan_result_markdown_row(report['warm_result'])]
    if report.get('cold_prewarm_result'):
        rows.append(scan_result_markdown_row(report['cold_prewarm_result']))
    rows.extend(scan_result_markdown_row(result) for result in report.get('cold_results', []))
    table = '\n'.join([
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] + ['---:'] * (len(headers) - 1)) + ' |',
        *rows,
    ])
    command_block = f'RUSTFLAGS="{BENCHMARK_RUSTFLAGS}" ' + ' '.join(command)
    sections = [
        '# Lance 10k Lookup Benchmark',
        f'- Dataset: `{report.get("dataset_path", "")}`',
        f'- Config: `{config_path}`',
        f'- Positions: `{positions_file}`',
        f'- Positions requested: `{report.get("positions_requested", "")}`',
        f'- Projected fields: `{report.get("projected_column_count", "")}`',
        '```bash\n' + command_block + '\n```',
        table,
    ]
    physical_plans = benchmark_physical_plans_markdown(report)
    if physical_plans:
        sections.append(physical_plans)
    return '\n\n'.join(sections)


def run_lookup_benchmark_10k(
    *,
    config_path: Path = BENCHMARK_CONFIG,
    positions_file: Path = BENCHMARK_10K_POSITIONS,
) -> tuple[dict[str, Any], str]:
    command = [
        'cargo', 'run', '--release',
        '--manifest-path', str(RUST_SANDBOX_MANIFEST),
        '--', 'bench',
        '--config', str(config_path),
        '--dataset-path', str(OUTPUT_DATASET),
        '--positions-file', str(positions_file),
    ]
    benchmark_env = dict(os.environ)
    benchmark_env['RUSTFLAGS'] = BENCHMARK_RUSTFLAGS
    command_text = f'RUSTFLAGS="{BENCHMARK_RUSTFLAGS}" ' + ' '.join(command)
    console.print(f'Running 10k lookup benchmark: {command_text}')
    result = subprocess.run(command, cwd=REPO, env=benchmark_env, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            'Rust Lance benchmark failed.\n'
            f'Command: {command_text}\n'
            f'stdout:\n{result.stdout}\n'
            f'stderr:\n{result.stderr}'
        )

    report_path = benchmark_report_path(config_path, OUTPUT_DATASET)
    report = json.loads(report_path.read_text())
    markdown = benchmark_markdown(report, config_path=config_path, positions_file=positions_file, command=command)
    md_path = report_path.with_name('benchmark_10k.md')
    md_path.write_text(markdown)
    display(Markdown(markdown))
    print(f'wrote {md_path}')
    return report, markdown


benchmark_10k_report, benchmark_10k_markdown = run_lookup_benchmark_10k()


In [ ]:
# ----- Persist Rust-backed inspect artifacts -----
def write_inspect_artifacts(report: dict[str, Any]) -> None:
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    json_path = REPORTS_DIR / 'inspect_rust.json'
    json_path.write_text(json.dumps(report, indent=2, sort_keys=True))

    schema_md = pd.DataFrame(report['schema']).to_markdown(index=False)
    index_md = pd.DataFrame(report['indexes']).to_markdown(index=False) if report['indexes'] else '| name |\n|---|\n'
    fragment_md = pd.DataFrame(report['fragments']).to_markdown(index=False)
    md_path = REPORTS_DIR / 'inspect_rust.md'
    md_path.write_text('\n\n'.join([
        f'# Rust Lance Inspect: {OUTPUT_DATASET}',
        '## Schema', schema_md,
        '## Indexes', index_md,
        '## Fragments', fragment_md,
    ]))
    print(f'wrote {json_path}')
    print(f'wrote {md_path}')


if inspect_report is not None:
    write_inspect_artifacts(inspect_report)
else:
    print('No inspect report to write.')
